# Module-Level Functions
> Module‑Level Functions (MLFs) sit one layer above the core API and extend its capabilities with higher‑order analytical operations. 

> While the core API provides the fundamental data structures and transformations, MLFs operate across this hierarchy to deliver tasks such as classification, clustering, similarity search, or other model‑driven analyses. They do not replace the core API; instead, they orchestrate and combine its primitives into reusable, semantically richer workflows that simplify advanced processing and make complex analytical patterns easier to express.

In [ ]:
#| hide
from nbdev import show_doc

In [ ]:
#| eval: false

from fhemb.utils.cutils import prepare_data, split_scale_data

In [ ]:
#| eval: false

show_doc(prepare_data, title_level=2, name="Prepare Data {#prepare_data}")

/Users/radned/.pyenv/versions/p311.fhemb/lib/python3.11/site-packages/fastcore/docscrape.py:259: UserWarning: Unknown section Processing Steps
  else: warn(msg)
/Users/radned/.pyenv/versions/p311.fhemb/lib/python3.11/site-packages/fastcore/docscrape.py:259: UserWarning: Unknown section Example
  else: warn(msg)


---

## Prepare Data {#prepare_data}

```python

def prepare_data(
    X0:ndarray, # Time series data for class 0 of shape (n_samples_0, n_timesteps, n_features) or (n_samples_0, n_timesteps).
    X1:ndarray, # Time series data for class 1 of shape (n_samples_1, n_timesteps, n_features) or (n_samples_1, n_timesteps).
)->tuple: # X: Concatenated and cleaned time series of shape (n_balanced_samples, n_timesteps, n_features).
y: Class labels (0 or 1) of shape (n_balanced_samples,).


```

*Prepare multi-class time series data for machine learning.*
Performs data cleaning, deduplication, NaN removal, and class balancing on two sets of time series data representing different classes.

In [ ]:
#| eval: false
show_doc(split_scale_data, title_level=2, name="Split and Scale Data {#split_scale_data}")

---

## Split and Scale Data {#split_scale_data}

```python

def split_scale_data(
    X:ndarray, # Time series of shape (n_samples, n_timesteps, n_features).
    y:ndarray, # Labels of shape (n_samples,).
    test_size:float=0.3, # Fraction of samples for test split.
    random_state:int=42, # Seed for stratified split.
    normalize:NoneType=None, # None, 'minmax', 'meanvar', or fitted scaler
): # (X_train, X_test, y_train, y_test, scaler)


```

*Stratified train-test split with optional normalization.*

`normalize` may be:

    - None: no scaling
    - 'minmax': fit MinMax scaler on train, apply to test
    - 'meanvar': fit MeanVariance scaler on train, apply to test
    - fitted scaler instance: reuse without refitting

## Train and Test Classifiers 

In [ ]:
#| eval: false

from fhemb.utils.cutils import calc_similarity, calc_accuracies

In [ ]:
#| eval: false

show_doc(calc_similarity, title_level=3, name="Calculate accuracy or f1 score")

---

### Calculate accuracy or f1 score

```python

def calc_similarity(
    y_test:ndarray, # Ground truth labels of shape (n_samples,).
    y_pred:ndarray, # Predicted labels of shape (n_samples,).
    sim_metric:str='accuracy', # Similarity metric to use. Options: 'accuracy', 'f1'. Default: 'accuracy'.
)->float: # Computed similarity score between 0 and 1.


```

*Calculate a similarity metric between test labels and predictions.*

In [ ]:
#| eval: false
show_doc(calc_accuracies, title_level=3, name="Train, test, and evaluate multiple classifiers {#calc_accuracies}")

/Users/radned/.pyenv/versions/p311.fhemb/lib/python3.11/site-packages/fastcore/docscrape.py:259: UserWarning: Unknown section Classifiers Tested
  else: warn(msg)


---

### Train, test, and evaluate multiple classifiers {#calc_accuracies}

```python

def calc_accuracies(
    X_train:matrix, # Training feature matrix of shape (n_samples_train, n_features).
    X_test:matrix, # Test feature matrix of shape (n_samples_test, n_features).
    y_train:ndarray, # Training labels of shape (n_samples_train,).
    y_test:ndarray, # Test labels of shape (n_samples_test,).
)->dict: # Dictionary mapping classifier names to accuracy scores.
Keys are formatted as 'classifier_variant' (e.g., 'knn_euclidean', 'svm_rbf', 'rotf').
Values are float accuracies between 0 and 1.


```

*Compute classification accuracies across multiple classifier algorithms.*

Tests multiple supervised learning algorithms (SVM, KNN, Random Forest, etc.) on
the provided train-test data and returns accuracies for each classifier-metric combination.

#### Classifier Variant Naming Convention

The return value is a dictionary mapping **classifier variants** to accuracy scores.

Each key is constructed as:
`<classifier_name>_`

where:

- KNN expands over **distance metrics**
- SVM expands over **kernel types**
- Other classifiers have a **single variant** (no suffix)

::: {.callout collapse="true" title= "Variant Naming Table"}
| Classifier name | Variants produced | Example keys | Notes |
|-----------------|-------------------|--------------|-------|
| `knn` | One variant per KNN metric in `KNN_METRICS` | `knn_euclidean`, `knn_cosine`, `knn_minkowski`, `knn_precomputed` | Metric name is appended directly |
| `svm` | One variant per SVM kernel in `SVM_KERNELS` | `svm_linear`, `svm_rbf`, `svm_poly`, `svm_sigmoid`, `svm_precomputed` | Kernel name is appended directly |
| `randf` | Single variant | `randf` | No suffix needed |
| `rotf` | Single variant | `rotf` | No suffix needed |
| `shapelet_rotationforest` | Single variant | `shapelet_rotationforest` | No suffix needed |
:::

##### Examples

If all classifiers and all metrics/kernels are evaluated, the result dictionary may look like:

```python
{
    "knn_euclidean": 0.83,
    "knn_cosine": 0.79,
    "knn_precomputed": 0.91,
    "svm_linear": 0.88,
    "svm_rbf": 0.92,
    "randf": 0.90,
    "rotf": 0.94,
    "shapelet_rotationforest": 0.96,
}
```


::: {.callout collapse="true" title= "Classifier-Based Similarity Functions - Summary Table"}
| Classifier name | Classifier | Key Hyperparameters | Input Requirements | Output | Strengths | Notes |
|-----------------|------------|---------------------|--------------------|--------|-----------|-------|
| `svm` | `sklearn.svm.SVC` | `kernel ∈ {linear, poly, rbf, sigmoid}`, `C` | Feature matrices (`X_train`, `X_test`), label vectors | Accuracy ∈ [0, 1] | Strong for linear or smooth nonlinear boundaries | Standard SVM; no probability outputs |
| `knn` | `KNeighborsClassifier (k=3)` | `metric ∈ KNN_METRICS`, `p` for Minkowski | Raw features or precomputed distance matrices | Accuracy ∈ [0, 1] | Works well with DTW/custom distances | Sensitive to scaling unless metric=`precomputed` |
| `randf` | `RandomForestClassifier` | `n_estimators=100`, `max_depth=None`, `random_state=42` | Standard feature matrices | Accuracy ∈ [0, 1] | Robust baseline for tabular embeddings | Fully grown trees; ensemble stabilizes variance |
| `rotf` | `RotationForest + DecisionTreeClassifier` | `n_estimators=100`, PCA‑based rotations | Standard feature matrices | Accuracy ∈ [0, 1] | Strong for time‑series and correlated features | Rotation Forest decorrelates via PCA subsets |
| `shapelet_rotationforest` | `ShapeletTransformClassifier + RotationForest` | `n_shapelet_samples=100`, `max_shapelets=10`, `batch_size=20`, `n_estimators=3` | Raw time‑series arrays | Accuracy ∈ [0, 1] | Captures discriminative subsequences | Best for raw time‑series; slower due to shapelet extraction |
:::


::: {.callout collapse="true" title= "KNN Distance Metrics for the `knn` classifier - Summary Table"}
| Metric name | Meaning / Distance Type | Equation | Input Requirements | Strengths | Notes |
|-------------|--------------------------|----------|--------------------|-----------|-------|
| `euclidean` | Standard L2 distance | $d(x,y) = \sqrt{\sum_i (x_i - y_i)^2}$ | Raw feature vectors | Stable, widely used, works well with normalized data | Equivalent to Minkowski with $p=2$ |
| `nan_euclidean` | NaN‑aware Euclidean distance | Same as Euclidean, computed only over non‑NaN dimensions | Raw features with possible NaNs | Robust when some features are missing | Ignores NaN dimensions; rescales by observed dims |
| `minkowski` | Generalized Lp distance | $d(x,y) = \left(\sum_i \|x_i - y_i\|^p\right)^{1/p}$ | Raw features; requires `p` | Flexible family including L1, L2, L∞ | `p` passed via `calc_knn_similarity(p=...)` |
| `cosine` | Cosine distance (1 − cosine similarity) | $d(x,y) = 1 - \frac{x \cdot y}{\|x\|\|y\|}$ | Raw feature vectors | Good for directional / high‑dimensional data | Scale‑invariant; undefined for zero vectors |
| `l1` | Manhattan (cityblock) distance | $d(x,y) = \sum_i \|x_i - y_i\|$ | Raw feature vectors | Robust to outliers; sparse‑friendly | Equivalent to Minkowski with $p=1$ |
| `precomputed` | User‑supplied distance matrix | *No equation — distances provided externally* | Requires full distance matrices for train/test | Allows DTW, soft‑DTW, custom metrics | `X_train` and `X_test` must be distance matrices |
:::

::: {.callout collapse="true" title= "SVM Kernel Functions for the `svm` classifier - Summary Table"}
| Kernel name | Meaning / Kernel Type | Equation | Strengths | Notes |
|-------------|------------------------|----------|-----------|-------|
| `linear` | Linear kernel (inner product) | $K(x, y) = x^\top y$ | Fast, stable, works well when classes are linearly separable | Equivalent to a linear classifier in feature space |
| `poly` | Polynomial kernel | $K(x, y) = (\gamma\, x^\top y + r)^d$ | Captures polynomial feature interactions | Degree $d$, scale $\gamma$, and offset $r$ correspond to `degree`, `gamma`, and `coef0` in `sklearn.svm.SVC` |
| `rbf` | Radial Basis Function (Gaussian) | $K(x, y) = \exp(-\gamma \|x - y\|^2)$ | Very powerful; handles nonlinear boundaries | $\gamma$ controls smoothness; default often works well |
| `sigmoid` | Sigmoid (tanh) kernel | $K(x, y) = \tanh(\gamma\, x^\top y + r)$ | Related to neural network activation functions | Not always positive‑definite; may require tuning |
| `precomputed` | User‑supplied kernel matrix | *No equation — kernel values provided externally* | Allows custom kernels, DTW kernels, similarity matrices | Input must be an $n_\text{train} \times n_\text{train}$ kernel matrix |
:::

## Diagnose via Dimensionality Reduction {#DiagnosticsMF}

::: {.callout-note title="Projection into a low-dimensional space"}
is based on **dimensionality reduction methods**: `MDS` and `TSNE` from `sklearn.manifold`, `PCA` from `sklearn.decomposition`, and `UMAP` from `umap-learn`. 

- *MDS* and *t‑SNE* internally operate on pairwise distances derived from the input samples, so they can be applied directly to distance‑based representations (e.g., *DTW distance matrices*). 
- *UMAP* can also consume **precomputed distances**, and unlike *t‑SNE* it attempts to preserve both local neighborhoods and larger‑scale manifold structure, making it well‑suited for nonlinear latent spaces derived from alignment metrics. 
- *PCA*, in contrast, assumes a similarity (covariance / inner‑product) structure. When the latent representation is defined only through distances or alignment scores, a **distance‑to‑similarity transformation** must be applied before *PCA* can be used.
:::

::: {.callout collapse="true" title="Dimensionality Reduction – Summary Table"}
| Projection method | Input Type | What It Optimizes | Strengths | Weaknesses | Typical Use |
|--------|------------|-------------------|-----------|------------|-------------|
| **MDS (Multidimensional Scaling)** | Distance matrix (precomputed) | Preserves pairwise distances | Stable, interpretable, works directly on distances | Can struggle with nonlinear structure | Visualizing global geometry of distance-based data |
| **t‑SNE** | Distance matrix (precomputed) | Preserves local neighborhoods | Excellent for clusters and nonlinear manifolds | Distorts global structure; parameter‑sensitive | Exploring clusters and local structure |
| **PCA** | Feature matrix (or similarity derived from distances) | Maximizes variance along orthogonal axes | Fast, deterministic, widely understood | Linear only; not naturally distance‑based | Baseline dimensionality reduction and comparison |
| **UMAP** | Distance matrix (precomputed) or feature matrix | Preserves local structure while retaining more global geometry | Fast, scalable, manifold‑aware; good balance of local/global structure | Sensitive to `n_neighbors` and `min_dist`; axes not interpretable | General‑purpose embedding for nonlinear structure with better global faithfulness |
:::